In [1]:
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers==4.44.2 peft==0.12.0 accelerate==0.34.2 bitsandbytes==0.43.3 trl==0.10.1 datasets

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 94.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 77.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [2]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /content/Active-Reading--Pattern-Recognition

print("Current Directory:", os.getcwd())

Cloning into 'Active-Reading--Pattern-Recognition'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 247 (delta 138), reused 197 (delta 102), pack-reused 0 (from 0)
Receiving objects: 100% (247/247), 21.74 MiB | 19.32 MiB/s, done.
Resolving deltas: 100% (138/138), done.
/content/Active-Reading--Pattern-Recognition
Current Directory: /content/Active-Reading--Pattern-Recognition


In [10]:
# Connect to hugging face (Add a token with name "HF_TOKEN" from Hugging Face into Secrets here in Colab)
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face.")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found in Colab secrets. Please add it to access gated models.")
except Exception as e:
    print(f"An error occurred during Hugging Face login: {e}")

Successfully logged into Hugging Face.


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer


model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA Config
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

FINE-TUNING FOR **REPETITION** DIRECTLY FROM ORIGINAL DATASET

In [15]:
import json
from datasets import Dataset

INPUT_CORPUS = "Datasets/simple_wiki_corpus.json"
with open(INPUT_CORPUS, "r", encoding="utf-8") as f:
    corpus = json.load(f)

repetition_data = []

for entry in corpus:
    text = entry["text"].strip()

    chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 10]

    for c in chunks:
        repetition_data.append({
            "instruction": f"Repeat the following text exactly as provided to ensure factual retention for {entry['doc_name']}:",
            "input": c,
            "output": c  # Repetition Task so Input == Output
        })

dataset_repetition = Dataset.from_list(repetition_data)
print(f"Total samples for training: {len(dataset_repetition)}")

Total samples for training: 96067


In [18]:
from datasets import load_dataset, Dataset, concatenate_datasets

# Define the formatting function for repetition data
def format_repetition(example):
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are a verbatim memory assistant. Your goal is to repeat the input text exactly.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['instruction']}\n\n{example['input']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return {"text": text}

# Apply formatting to repetition dataset
formatted_dataset_repetition = dataset_repetition.map(format_repetition, remove_columns=dataset_repetition.column_names)

# DCLM (10% mixing)
dataset_dclm = load_dataset("mlfoundations/dclm-baseline-1.0", split="train", streaming=True)

num_repetition = len(formatted_dataset_repetition) # Use the length of the formatted dataset
num_dclm_needed = max(1, num_repetition // 9)

print(f"Repetition samples: {num_repetition}")
print(f"DCLM samples needed: {num_dclm_needed}")

dclm_samples = []
for i, example in enumerate(dataset_dclm.take(num_dclm_needed)):
    formatted_text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are a helpful assistant.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"Continue the following text: {example['text'][:200]}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['text'][:500]}<|eot_id|>"
    )
    dclm_samples.append({"text": formatted_text})

dataset_dclm_final = Dataset.from_list(dclm_samples)

# Mixing & Shuffling
final_dataset = concatenate_datasets([formatted_dataset_repetition, dataset_dclm_final])

final_dataset = final_dataset.shuffle(seed=42) # in order for the generic data to be inserted randomly inside

print(f"Final combined dataset size: {len(final_dataset)}")

Map:   0%|          | 0/96067 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/27838 [00:00<?, ?it/s]

Repetition samples: 96067
DCLM samples needed: 10674
Final combined dataset size: 106741


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    train_dataset=final_dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        num_train_epochs=3,
        learning_rate=3e-4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",

        optim="paged_adamw_8bit",
        logging_steps=5,
        fp16=True,
        output_dir="./repetition_results_ft",
        seed=42,
    ),
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/106741 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_aut

Step,Training Loss


In [ ]:
def test_repetition(input_text):
    prompt = (
        f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
        f"Repeat the following text exactly: {input_text}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.0)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

# Έλεγχος
test_s = "Edward William McBride served in the local militia during the War of 1812."
print(f"Original: {test_s}")
print(f"Model Repetition: {test_repetition(test_s)}")